# Gold Layer - Sales by Hour View

## Purpose
Provide real-time hourly sales analysis (0-23 hours) to identify peak business hours without materializing data.

## Type
**SQL View** (not a materialized table)

## Output
* **View:** `big_data.gold.vw_sales_by_hour`
* **Rows:** 24 (hours 0-23)

## Usage
```sql
SELECT * FROM big_data.gold.vw_sales_by_hour
WHERE order_hour_of_day BETWEEN 9 AND 17;  -- Business hours
```

In [0]:
%sql
-- Create or replace view: vw_sales_by_hour

CREATE OR REPLACE VIEW big_data.gold.vw_sales_by_hour AS
SELECT 
  o.order_hour_of_day,
  COUNT(DISTINCT o.order_id) AS total_orders,
  COUNT(op.product_id) AS total_items,
  ROUND(SUM(p.price_usd), 2) AS estimated_revenue_usd,
  ROUND(AVG(p.price_usd), 2) AS avg_item_price_usd,
  ROUND(SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) * 100.0 / COUNT(op.product_id), 2) AS reorder_rate
FROM big_data.silver.orders o
JOIN big_data.silver.order_products op ON o.order_id = op.order_id
JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
GROUP BY o.order_hour_of_day
ORDER BY o.order_hour_of_day;

In [0]:
%sql
-- Verify view exists and preview top hours by orders
SELECT * FROM big_data.gold.vw_sales_by_hour
ORDER BY total_orders DESC
LIMIT 5;